# L04 · 声音的骨架 — 正弦波三要素

**目标**：掌握 sin/cos 的形状，以及振幅 A、频率 f、相位 φ 各控制什么。

**为什么对 Aurora 重要**：`aurora.audio.sine` 直接实现 `A·sin(2πft+φ)`；Week 1 Day 2 要从零重写这个函数并用 `assert np.allclose` 对答案。

## 本课剧情：操作波形调音台

三个参数完全决定一个正弦波的形状：A 控制幅度，f 控制周期快慢，φ 控制从哪个角度开始。把它们分开来改，波形的变化就一目了然。

## 1. sin 与 cos

## 实验入口：角度、坐标和旋转

sin 和 cos 描述同一个圆周运动的两个分量：cos 比 sin 超前四分之一个周期。观察两条曲线的峰值、零点、谷值的位置关系，确认 cos(0)=1 而 sin(0)=0。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 2*np.pi, 400)
plt.plot(t, np.sin(t), label='sin')
plt.plot(t, np.cos(t), label='cos')
plt.legend(); plt.title('sin & cos'); plt.grid(True, alpha=.3); plt.show()

## 动手观察：四个特殊角度

0、π/2、π、3π/2 这四个角度的 cos 和 sin 值都是 0 或 ±1，可以直接心算验证。运行后比对 `z.real` 和 `z.imag`，确认输出与预期完全一致。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：遍历频率、振幅和相位

采样率 16 Hz、时长 0.5 s 只产生 8 个采样点，数字少到可以逐个检查：freq 加倍时相邻样本变化幅度翻倍，amplitude 加倍时输出的 min/max 同比例放大。

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## 2. 一个正弦波 = `A · sin(2π·f·t + φ)`

- **A 振幅** = 多响（把 `[-1, 1]` 的输出拉伸到 `[-A, A]`）
- **f 频率** = 多高，Hz（每秒完整振荡次数，决定音高）
- **φ 相位** = 起始角度偏移，单位弧度（φ=π/2 时波形从峰值开始）

三个参数互相正交：A 只缩放幅度、f 只改变时间轴上的振荡速率、φ 只偏移起始角度——改任何一个不影响另外两个。这种正交设计让合成复杂波形（如和弦、MFCC filterbank 的各个分量）时可以独立调节每个成分，而不需要重新校准其他参数。

## 3. ✏️ 实现 `sinusoid(t, A, f, phi)`

**推理路线**：
1. `t` 是时间数组（秒），`f` 是每秒完整周期数（Hz）。一个完整周期对应 `2π` 弧度，所以时刻 `t` 处的角度是 `2π·f·t`。
2. 加上初相位 `phi`（弧度），得到完整角度序列 `2π·f·t + phi`。
3. `np.sin(...)` 把角度映射到 `[-1, 1]`；乘以 `A` 把值域扩展到 `[-A, A]`。注意顺序：先算角度，再一次性乘 A，避免多次标量乘法。

**参考输入输出**：`A=2, f=1, phi=0, t=[0, 0.25, 0.5, 0.75]` → `[0, 2, 0, -2]`（一个周期的四个采样点）

<details><summary>点击查看参考实现</summary>

```python
return A * np.sin(2 * np.pi * f * t + phi)
```

</details>

### 写代码前，先把变量表补完整

写 `sinusoid` 前明确三件事：
- 输入：`t`（时间数组）、`A`（振幅）、`f`（频率 Hz）、`phi`（初相位，默认 0）
- 关键步骤：计算相位序列 `2π·f·t + phi`，乘以 A
- 返回：与 `t` 同形状的浮点数组，值域在 `[-A, A]`

In [ ]:
def sinusoid(t, A, f, phi=0.0):
    # ✏️ TODO: 返回正弦波
    return None  # ← 改这里


In [ ]:
import numpy as np
t = np.linspace(0, 1, 8, endpoint=False)
y = sinusoid(t, 2.0, 1.0, 0.0)
assert np.allclose(y, 2*np.sin(2*np.pi*1.0*t)), '应等于 2·sin(2πt)'
print('✅ 通过：你能造任意振幅/频率/相位的正弦波了。')

**🔗 Aurora 连接**：`aurora.audio.sine` 就是它；Week 1 Day 2 你已经重写过。下一课进入复数——FFT 的输出类型。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | 过零次数≈{zero_crossings}')


## 参数实验：只改一个旋钮

把 `f` 从 1 改到 4，观察相同 `t` 范围内过零点数量变为原来的 4 倍。再把 `phi` 改到 `np.pi/2`，确认波形从余弦形状（先到峰值）开始。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | max={x.max():.1f}')


## 本课收束

现在可以用 `sinusoid(t, A, f, phi)` 生成任意振幅、频率、相位的正弦波，并用 `assert np.allclose` 验证数值是否正确。这正是 `aurora.audio.sine` 的核心实现；Week 1 Day 2 你已经手写过一遍并对答案。下一节引入复数表示，`e^{jθ}` 的实部和虚部正好是今天的 cos 和 sin，是理解 FFT 输出的基础。

In [ ]:
# 小检查：同一个角度，同时看 cos、sin、复数
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')
